In [50]:
import pandas as pd 
import ast 
import nltk
#Para poder delimitar las oraciones y despues tokenizar
nltk.download("punkt_tab")
#Para poder identificar las stopwords
from nltk.corpus import stopwords 
nltk.download("stopwords")
#Para identificar las puntuaciones 
import string
import numpy as np
#Para aplicar lematizacion
import spacy 
#Para tratar los tildes 
import unicodedata
from sklearn.feature_extraction.text import TfidfVectorizer


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\semab\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\semab\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df = pd.read_csv("ted_talks_es.csv")
df.head()

,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,native_lang,available_lang,comments,duration,topics,related_talks,url,description,transcript
0,1,Al Gore sobre cómo evitar la crisis climática,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523396,2006-02-25,2006-06-27,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272.0,977,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,Con el mismo humor y humanidad que irradió en ...,Muchas gracias Chris. Y es en verdad un gran h...
1,7,"David Pogue dice ""La Simplicidad Vende""",David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920803,2006-02-24,2006-06-27,TED2006,en,"['ar', 'bg', 'de', 'el', 'en', 'es', 'fa', 'fr...",124.0,1286,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,"El columnista del New York Times, David Pogue,...","Hola contestadora automática, mi vieja amiga. ..."
2,53,Un recorrido por la renovación urbana de la ma...,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664029,2006-02-26,2006-06-27,TED2006,en,"['ar', 'bg', 'bn', 'ca', 'cs', 'de', 'en', 'es...",219.0,1116,"['MacArthur grant', 'activism', 'business', 'c...",{1041: '3 stories of local eco-entrepreneurshi...,https://www.ted.com/talks/majora_carter_greeni...,"En una charla altamente emotiva, la activista ...","Si están presentes aquí hoy, y estoy muy conte..."
3,66,Ken Robinson dice que las escuelas matan la cr...,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65052534,2006-02-25,2006-06-27,TED2006,en,"['af', 'ar', 'az', 'be', 'bg', 'bn', 'ca', 'cs...",4931.0,1164,"['children', 'creativity', 'culture', 'dance',...","{865: 'Bring on the learning revolution!', 173...",https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson plantea de manera entretenida...,"Buenos días. ¿Cómo están? Ha sido increíble, ¿..."
4,92,Hans Rosling nos muestra las mejores estadísti...,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501766,2006-02-22,2006-06-27,TED2006,en,"['ar', 'az', 'bg', 'bn', 'bs', 'cs', 'da', 'de...",628.0,1190,"['Africa', 'Asia', 'Google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,Una manera única de presentar datos. Con la en...,"Hace unos 10 años, emprendí la tarea de enseña..."


In [3]:
##Es un string ,necesitmaos que sea una lista para usar explode
type(df.topics[0])

str

In [4]:
### Citar de donde saque para usar literal eval 
### https://stackoverflow.com/questions/1894269/how-to-convert-string-representation-of-list-to-a-list

In [5]:
##Transformamos de string a lista
df["topics"] = df.topics.apply(lambda x : ast.literal_eval(x))

In [6]:
df.topics

0       [alternative energy, cars, climate change, cul...
1       [computers, entertainment, interface design, m...
2       [MacArthur grant, activism, business, cities, ...
3       [children, creativity, culture, dance, educati...
4       [Africa, Asia, Google, demo, economics, global...
                              ...                        
3916    [life, society, immigration, humanity, self, p...
3917    [TED-Ed, education, animation, fashion, climat...
3918    [life, emotions, communication, stigma, psycho...
3919    [coronavirus, pandemic, epidemiology, virus, b...
3920    [TED-Ed, education, history, animation, intell...
Name: topics, Length: 3921, dtype: object

In [7]:
#Expande la lista y crea una fila por topico
explode  = df.explode("topics")
explode.head(4)

,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,native_lang,available_lang,comments,duration,topics,related_talks,url,description,transcript
0,1,Al Gore sobre cómo evitar la crisis climática,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523396,2006-02-25,2006-06-27,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272.0,977,alternative energy,"{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,Con el mismo humor y humanidad que irradió en ...,Muchas gracias Chris. Y es en verdad un gran h...
0,1,Al Gore sobre cómo evitar la crisis climática,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523396,2006-02-25,2006-06-27,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272.0,977,cars,"{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,Con el mismo humor y humanidad que irradió en ...,Muchas gracias Chris. Y es en verdad un gran h...
0,1,Al Gore sobre cómo evitar la crisis climática,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523396,2006-02-25,2006-06-27,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272.0,977,climate change,"{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,Con el mismo humor y humanidad que irradió en ...,Muchas gracias Chris. Y es en verdad un gran h...
0,1,Al Gore sobre cómo evitar la crisis climática,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523396,2006-02-25,2006-06-27,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272.0,977,culture,"{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,Con el mismo humor y humanidad que irradió en ...,Muchas gracias Chris. Y es en verdad un gran h...


In [8]:
#Los topicos que hay 
#topicos = list(explode.topics.unique())
#for x in topicos : 
    #print(x)

In [9]:
#Necesitamos saber el tema que va a tratr y seleccionar aquellos temas que se relacionan
#Seleccionamos todas las charlas que hablan de inteligencia artificial. 
AI = explode[explode.topics == "AI"]
AI

,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,native_lang,available_lang,comments,duration,topics,related_talks,url,description,transcript
104,125,"Jeff Hawkins en ""Cómo la ciencia del cerebro c...",Jeff Hawkins,{0: 'Jeff Hawkins'},"{0: ['computer designer', 'brain researcher']}","{0: ""Jeff Hawkins pioneered the development of...",1683289,2003-02-02,2007-05-21,TED2003,en,"['ar', 'bg', 'da', 'de', 'el', 'en', 'es', 'fa...",244.0,1211,AI,"{184: '3 clues to understanding your brain', 2...",https://www.ted.com/talks/jeff_hawkins_how_bra...,"Creador del Treo, Jeff Hawkins, nos anima a ve...","Yo hago dos cosas, diseño computadora móviles ..."
147,165,"Hod Lipson construye robots ""autoconscientes""",Hod Lipson,{0: 'Hod Lipson'},{0: ['roboticist']},{0: 'Hod Lipson works at the intersection of e...,1465747,2007-03-06,2007-10-11,TED2007,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",242.0,378,AI,"{195: 'The sticky wonder of gecko feet', 146: ...",https://www.ted.com/talks/hod_lipson_building_...,Hod Lipson muestra unos cuantos de sus pequeño...,"Entonces, ¿dónde están los robots? Nos han dic..."
295,355,Rodney Brooks dice que los robots invadirán nu...,Rodney Brooks,{0: 'Rodney Brooks'},{0: ['roboticist']},{0: 'Rodney Brooks builds robots based on biol...,692492,2003-02-02,2008-09-29,TED2003,en,"['ar', 'bg', 'cs', 'en', 'es', 'fr', 'he', 'it...",85.0,1127,AI,"{195: 'The sticky wonder of gecko feet', 165: ...",https://www.ted.com/talks/rodney_brooks_robots...,"En esta charla profética, del 2003, el experto...",De lo que quiero hablarles hoy es de cómo veo ...
628,820,Dennis Hong: Mis siete especies de robots.,Dennis Hong,{0: 'Dennis Hong'},{0: ['roboticist']},{0: 'Dennis Hong is the founder and director o...,2248065,2009-09-10,2010-04-07,TEDxNASA,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",189.0,955,AI,{280: 'Robots inspired by cockroach ingenuity'...,https://www.ted.com/talks/dennis_hong_my_seven...,"En TEDxNASA, Dennis Hong presenta siete robots...",El primer robot del que voy a hablar se llama ...
709,932,"Peter Molyneux hace una demostración de Milo, ...",Peter Molyneux,{0: 'Peter Molyneux'},{0: ['game changer']},{0: 'The head of Microsoft\'s European games d...,854065,2010-07-09,2010-08-18,TEDGlobal 2010,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",211.0,655,AI,"{799: 'Gaming can make a better world', 816: '...",https://www.ted.com/talks/peter_molyneux_meet_...,"Peter Molyneux hace una demostración de Milo, ...","Cuando vi una tecnología denominada Kinect, se..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3654,49131,El peligro de la IA es más extraño de lo que p...,Janelle Shane,{0: 'Janelle Shane'},{0: ['ai researcher']},{0: 'While moonlighting as a research scientis...,406152,2019-04-15,2019-10-22,TED2019,en,"['ar', 'cs', 'en', 'es', 'fa', 'fr', 'he', 'hu...",26.0,628,AI,"{36479: 'How to keep human bias out of AI', 40...",https://www.ted.com/talks/janelle_shane_the_da...,El peligro de la inteligencia artificial no es...,La inteligencia artificial es conocida por tra...
3658,50986,"El mito griego de Talos, el primer robot",Adrienne Mayor,{0: 'Adrienne Mayor'},NaN,NaN,1261885,2019-10-24,2019-10-24,TED-Ed,en,"['ar', 'de', 'el', 'en', 'es', 'fa', 'fr', 'he...",NaN,223,AI,{46592: 'The myth of Jason and the Argonauts '...,https://www.ted.com/talks/adrienne_mayor_the_g...,Mira la lección completa en https://ed.ted.com...,"Hefesto, el dios de la tecnología, se encontra..."
3663,50637,El potencial médico de la IA y los metabolitos,Leila Pirhaji,{0: 'Leila Pirhaji'},{0: ['biotech entrepreneur']},{0: 'Leila Pirhaji uses artificial intelligenc...,1396187,2019-04-15,2019-10-29,TED2019,en,"['ar', 'cs', 'de', 'en', 'es', 'fa', 'fr', 'gu...",7.0,314,AI,{20008: 'How AI is making it easier to diagnos...,https://www.ted.com/talks/leila_pirhaji_the_me...,Muchas enfermedades son causadas por metabolit...,"En 2003, cuando se

In [10]:
### NANDO = [1,2,5,6]
#YO = [3,4,7]
#JUNTO = [8,9]

In [11]:
#MUESTRA DE COMO ES EL TEXTO DE UNA CHARLA solo primeros 1000 caracteres
AI.transcript.iloc[0][0:1000]

'Yo hago dos cosas, diseño computadora móviles y estudio cerebros. Y la charla de hoy se trata de cerebros y, ¡Si! En algún lugar tengo un fan de cerebros. (Risas) Voy a, si me pueden poner la primera diapositiva, y verán el titulo de mi platica y mis dos afiliaciones. Así que de lo que voy a hablar, es el porqué no tenemos una buena teoría acerca del cerebro, porque es importante que desarrollemos una y qué podemos hacer al respecto. Intentaré hacer eso en 20 minutos. Tengo dos afiliaciones. La mayoría de ustedes me conocen por mis días de Palm y Handspring pero también administro un instituto de investigación científica sin fines de lucro. llamado el Instituto Redwood de Neurociencia en Menlo Park, y estudiamos neurociencia teórica, y estudiamos como funciona la neo-corteza. Voy a hablar acerca de todo eso. Tengo sólo una diapositiva sobre mi otra vida, las computadoras, y es ésta. Estos son algunos de los productos que desarrollé en los últimos 20 años, empezando desde la muy origin

In [12]:
##Todos las charlas 
charlas = list(AI.transcript)
#CANTIDAD DE CHARLAS A PROCESAR
len(charlas)

74

## Preprocesamiento de texto

In [14]:
#tokenizacion 
#Tokenizo cada charla
tokenization = [nltk.word_tokenize(charla) for charla in charlas]
tokenization[0][0:6]

['Yo', 'hago', 'dos', 'cosas', ',', 'diseño']

### Saqué la informacion desde https://www.geeksforgeeks.org/python/how-to-remove-string-accents-using-python-3/

In [16]:
def quitar_tildes(texto): 
   # Normalizamos el texto
    normalizado= unicodedata.normalize('NFKD', texto)  
    
    final = ''.join([letra for letra in normalizado if not unicodedata.combining(letra)])  
    return final 

In [17]:
#Parte limpieza de stopwords,puntuacion ,tildes y signos de interrogacion
limpio = []
stopword =stopwords.words("spanish")
puntuacion = string.punctuation 
contador = 0
for charla in tokenization : 
    #Por cada charla revisamos si sus token tienen stopwords o puntuacion
    #Eliminamos signos de interrogacion(¿?) que estaban pegados a las palabras
    charla_limpia =[]
    for palabra in charla : 
        palabra_limpia = ""
        for letra in palabra : 
            #puntuacion no tiene "¡" asi que lo agrego manual
            if letra not in puntuacion and letra != "¡":
                palabra_limpia+=letra
        #Aqui termina de procesar palabra 
        #Quito los tildes y añado en minunscula 
        palabra_limpia = quitar_tildes(palabra_limpia)
        charla_limpia.append(palabra_limpia.lower())
    charla_limpia = [token for token in charla_limpia if token not in stopword and token not in puntuacion]
    limpio.append(charla_limpia)
    


In [18]:
## Le quitamos los tildes para reducir el tamaño de la matriz y "quitar ruido"

In [19]:
limpio[0][0:6]

['hago', 'dos', 'cosas', 'diseno', 'computadora', 'moviles']

## La informacion para aplicar lematizacion a codigo la sequé de : https://localhorse.net/article/como-lematizar-en-python 

## Para que se pueda ejecutar se debe descargar la libreria de spacy y se debe ejecutar "python -m spacy download es_core_news_sm" en la terminal  en la ruta donde se tenga a python

# Ademas de https://stackoverflow.com/questions/62268302/spacy-es-core-news-sm-model-not-loading para solucionar error de importacion 

In [32]:
#Aplico lematizacion para reducir tamaño de matriz,lo que hace es que reduce la palabra a su forma base y asegura que sea válida
modelo = spacy.load("es_core_news_sm")
charla_final = []
contador=0
for charla in limpio :
    if contador<1:
        #Me pide string asi que junto todo 
        charla_unida = " ".join(charla)
        doc = modelo(charla_unida)
        lematizado =[token.lemma_ for token in doc]
        charla_final.append(lematizado)

    
    

In [38]:
limpio[0][0:10]

['hago',
 'dos',
 'cosas',
 'diseno',
 'computadora',
 'moviles',
 'estudio',
 'cerebros',
 'charla',
 'hoy']

In [42]:
#Por problemas de gramatica no sale a cuenta aplicar lematizacion por el significado que se otorga
charla_final[0] [0:10]

['hacer',
 'dos',
 'cosa',
 'diseno',
 'computadoro',
 'movil',
 'estudio',
 'cerebro',
 'char él',
 'hoy']

# Para la aplicacion de quitar terminos no relevante sy los metodos de TfidfVectorizer la saqué de :https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [186]:
#Aplicamos TfidfVectorizer ,dejo las palabras que aparezcan en al menos 3 charlas
tfid = TfidfVectorizer(min_df=3)
#Como tfid acepta solo string ,unimos charlas 
charlas_unidas =[" ".join(charla) for charla in limpio]
tfid_matriz = tfid.fit_transform(charlas_unidas)

In [188]:
#Cada fila es una documento  (reporte vocabulario final)
tfid_matriz

<74x3651 sparse matrix of type '<class 'numpy.float64'>'
	with 31911 stored elements in Compressed Sparse Row format>

In [190]:
#Cantidad de palabras unicas final (vocabulario)
len(tfid.vocabulary_)

3651

# El preprocesamiento es valido ,ya que primero ,reducimos tamaño ,dejando todo en minuscula,eliminamos las tildes para que no hayan problemas con que a algunas palabras no le pusieron y que otras si ,entonces se nos generen mas columnas innecesariamente, ademas eliminamos signos de puntuacion y stopwords.Ademas dejamos las palabras que se repiten en al menos 3 charlas ,para eliminar las demas que son raras y así eliminamos ruido

In [197]:
#Transponemos la matriz para seguir normativa del curso,donde los documentosson las columnas y los terminos son las filas
matriz_final = tfid_matriz.T

# 4. Construir matriz textual 

In [206]:
#Mostramos como se organiza la matriz
df = pd.DataFrame(matriz_final.toarray(), index=tfid.get_feature_names_out())
df

,0,1,2,3,4,5,6,7,8,9,...,64,65,66,67,68,69,70,71,72,73
000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.026025,0.000000,0.0,0.037766,0.0,0.012784
10,0.000000,0.0,0.010931,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.014277,...,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.032044,0.0,0.065083
100,0.019739,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.013432,0.047271,...,0.019657,0.068439,0.0,0.0,0.000000,0.000000,0.0,0.035367,0.0,0.011972
1000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000
11,0.000000,0.0,0.017896,0.0,0.042535,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
xix,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000
xx,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000
xxi,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.019372,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000
york,0.000000,0.0,0.000000,0.0,0.000000,0.050744,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000


# Dimension de la matriz 

In [209]:
df.shape

(3651, 74)

## La matriz fue construida desde el proceso donde aplicamos TfidfVectorizer,donde lo que hace es darle un peso a cada valor,seugun su frecuencia y las penaliza si son muy frecuentes

# La entrada $A_{ij}$ representa que tan importante es la palabra i en la charla j